# Atelier pratique Data Analyst — Semaine 1
## Analyse commerciale : ventes, clients et produits

**Objectif métier :** aider une direction commerciale à comprendre les ventes, repérer les produits et périodes performants, et proposer des actions concrètes.

Ce notebook est conçu comme un véritable mini-cas d'entreprise. Vous allez travailler avec **Python, pandas, SQL (SQLite), Matplotlib et Seaborn**.

### Votre mission
Vous êtes Data Analyst junior dans une entreprise fictive de vente en ligne. La direction vous demande un diagnostic des ventes sur une année.

À la fin, vous devrez pouvoir répondre à des questions telles que :
- Quel est le chiffre d'affaires et comment évolue-t-il ?
- Quels produits et quelles catégories contribuent le plus aux ventes ?
- Quels segments de clients sont les plus intéressants ?
- Existe-t-il des anomalies ou des problèmes de qualité des données ?
- Quelles recommandations proposer à la direction ?

### Organisation
- Niveau : intermédiaire progressif, avec rappels.
- Durée indicative : 2 à 4 heures, à répartir sur la semaine.
- Données : données fictives générées directement dans le notebook (aucun téléchargement nécessaire).
- Volume : environ 5 000 commandes, 1 200 clients et 30 produits.
- SQL : une base SQLite sera créée localement à l'exécution.

**Conseil :** essayez de répondre avec vos propres mots avant de regarder les pistes. Une bonne analyse ne consiste pas seulement à produire un graphique : elle doit expliquer ce que les résultats signifient pour le métier.

## 0. Préparer l'environnement

### Rappel — bibliothèques
- `pandas` : importer, nettoyer, transformer et analyser des tableaux.
- `numpy` : calcul numérique et génération de données.
- `matplotlib` : personnaliser les graphiques.
- `seaborn` : produire rapidement des visualisations statistiques.
- `sqlite3` : interroger une base SQL sans installer de serveur.

Si une bibliothèque manque dans votre environnement, décommentez la ligne d'installation adaptée.

In [ ]:
# Si nécessaire, exécutez cette cellule :
# %pip install pandas numpy matplotlib seaborn

In [ ]:
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
sns.set_theme(style="whitegrid")

print("Environnement prêt.")
print("pandas :", pd.__version__)

## 1. Découverte du jeu de données

### Contexte et dictionnaire des données

Nous disposons de trois tables :
- **customers** : informations sur les clients.
- **products** : catalogue des produits.
- **orders** : lignes de commande (une ligne représente un produit acheté dans une commande).

| Table | Champ | Signification |
|---|---|---|
| customers | customer_id | Identifiant client |
| customers | region | Région du client |
| customers | customer_segment | Segment : Particulier, Professionnel, Premium |
| products | product_id | Identifiant produit |
| products | category | Catégorie du produit |
| products | unit_price | Prix unitaire catalogue |
| orders | order_id | Identifiant de commande |
| orders | order_date | Date de commande |
| orders | customer_id | Client ayant commandé |
| orders | product_id | Produit acheté |
| orders | quantity | Quantité achetée |
| orders | unit_price | Prix unitaire réellement facturé |
| orders | discount_rate | Taux de remise |
| orders | sales_channel | Canal de vente |

**Attention métier :** le chiffre d'affaires net d'une ligne est calculé ici comme `quantity × unit_price × (1 - discount_rate)`. Il ne faut pas confondre chiffre d'affaires, nombre de commandes et nombre d'articles vendus.

### 1.1 Générer les données

Les données sont synthétiques mais construites pour ressembler à un contexte commercial. Quelques défauts seront volontairement introduits pour vous permettre de pratiquer le contrôle qualité.

In [ ]:
rng = np.random.default_rng(42)

# Référentiel clients
n_customers = 1200
regions = ["Île-de-France", "Auvergne-Rhône-Alpes", "Occitanie",
           "Nouvelle-Aquitaine", "Hauts-de-France", "PACA", "Bretagne"]
segments = ["Particulier", "Professionnel", "Premium"]

customers = pd.DataFrame({
    "customer_id": [f"C{i:04d}" for i in range(1, n_customers + 1)],
    "region": rng.choice(regions, n_customers, p=[.25, .17, .14, .13, .12, .11, .08]),
    "customer_segment": rng.choice(segments, n_customers, p=[.62, .25, .13]),
    "signup_date": pd.to_datetime("2023-01-01") + pd.to_timedelta(
        rng.integers(0, 900, n_customers), unit="D")
})

# Catalogue produits
categories = {
    "Informatique": ["Ordinateur", "Écran", "Clavier", "Souris", "Casque"],
    "Mobilier": ["Bureau", "Chaise", "Lampe", "Étagère", "Fauteuil"],
    "Papeterie": ["Cahier", "Stylo", "Classeur", "Agenda", "Papier"],
    "Téléphonie": ["Smartphone", "Coque", "Chargeur", "Écouteurs", "Support"],
    "Accessoires": ["Sac", "Gourde", "Câble", "Adaptateur", "Housse"],
    "Électroménager": ["Cafetière", "Bouilloire", "Aspirateur", "Grille-pain", "Mixeur"]
}
product_rows = []
pid = 1
for category, names in categories.items():
    for name in names:
        product_rows.append({
            "product_id": f"P{pid:03d}",
            "product_name": name,
            "category": category,
            "unit_price": round(float(rng.uniform(8, 650)), 2)
        })
        pid += 1
products = pd.DataFrame(product_rows)

# Lignes de commandes : environ 5 000
n_orders = 5000
order_dates = pd.to_datetime("2025-01-01") + pd.to_timedelta(
    rng.integers(0, 365, n_orders), unit="D")
order_customer = rng.choice(customers["customer_id"], n_orders)
order_product = rng.choice(products["product_id"], n_orders)
product_price = products.set_index("product_id")["unit_price"]
orders = pd.DataFrame({
    "order_id": [f"CMD{i:05d}" for i in range(1, n_orders + 1)],
    "order_date": order_dates,
    "customer_id": order_customer,
    "product_id": order_product,
    "quantity": rng.choice([1, 2, 3, 4, 5, 6], n_orders, p=[.48, .25, .13, .08, .04, .02]),
    "unit_price": [product_price[p] for p in order_product],
    "discount_rate": rng.choice([0, .05, .10, .15, .20], n_orders,
                                p=[.45, .20, .20, .10, .05]),
    "sales_channel": rng.choice(["Site web", "Marketplace", "Téléphone"], n_orders,
                                p=[.62, .28, .10])
})

# Défauts volontaires pour les exercices de qualité
orders.loc[rng.choice(orders.index, 18, replace=False), "quantity"] = np.nan
orders.loc[rng.choice(orders.index, 12, replace=False), "sales_channel"] = None
orders = pd.concat([orders, orders.sample(8, random_state=7)], ignore_index=True)

print("customers :", customers.shape)
print("products  :", products.shape)
print("orders    :", orders.shape)
display(orders.head())
display(customers.head())
display(products.head())

### Exercices 1 — prise en main

**1.** Combien de lignes et de colonnes contient chaque table ? Que représente une ligne dans `orders` ?

**2.** Affichez les types de données de chaque table. La colonne `order_date` est-elle bien reconnue comme une date ?

**3.** Affichez les statistiques descriptives des colonnes numériques de `orders`. Repérez une valeur qui vous semble étonnante et expliquez pourquoi.

**À vous :** rédigez ici vos premières observations.

In [ ]:
# Exercice 1.1 — dimensions
# À compléter


In [ ]:
# Exercice 1.2 — types et informations générales
# À compléter


In [ ]:
# Exercice 1.3 — statistiques descriptives
# À compléter


## 2. Qualité et préparation des données

### Mini-rappel — les contrôles essentiels
Avant de calculer des indicateurs, vérifiez :
1. les valeurs manquantes (`isna`, `isnull`) ;
2. les doublons (`duplicated`, `drop_duplicates`) ;
3. les types (`dtypes`, `to_datetime`, `astype`) ;
4. les valeurs impossibles ou incohérentes (quantité négative, remise supérieure à 100 %, prix nul) ;
5. les clés de jointure entre les tables.

**Point de vigilance :** ne supprimez pas automatiquement toutes les lignes incomplètes. Il faut comprendre leur importance et justifier le traitement choisi.

### Exercices 2 — audit qualité

**4.** Calculez le nombre et le pourcentage de valeurs manquantes par colonne dans `orders`.

**5.** Recherchez les doublons exacts. Combien en trouvez-vous ? Supprimez-les dans une copie de travail, sans modifier le jeu initial.

**6.** Vérifiez les valeurs aberrantes ou incohérentes dans `quantity`, `unit_price` et `discount_rate`.

**7.** Proposez un traitement justifié pour les valeurs manquantes de quantité et de canal de vente. Créez `orders_clean`.

**À vous :** quelles limites les défauts de qualité pourraient-ils avoir sur les indicateurs commerciaux ?

In [ ]:
# Exercice 2.1 — valeurs manquantes
# À compléter


In [ ]:
# Exercice 2.2 — doublons
# À compléter


In [ ]:
# Exercice 2.3 — contrôle des valeurs
# À compléter


In [ ]:
# Exercice 2.4 — créer une copie nettoyée
# À compléter : documentez votre choix dans une cellule Markdown.


## 3. Analyse exploratoire avec pandas

### Mini-rappel — opérations pandas utiles

- Filtrer : `df.loc[condition]`
- Trier : `sort_values()`
- Compter : `value_counts()` ou `nunique()`
- Agréger : `groupby(...).agg(...)`
- Créer une variable : `df["nouvelle_colonne"] = ...`
- Fusionner : `merge(..., on="cle", how="left")`
- Dates : `dt.month`, `dt.quarter`, `dt.day_name()`, `dt.to_period("M")`

Une **métrique** mesure un phénomène (CA, quantité, nombre de clients). Un **KPI** est un indicateur suivi pour piloter un objectif.

### Exercices 3 — construire les indicateurs

À partir de `orders_clean` :

**8.** Créez une colonne `revenue_net` correspondant au chiffre d'affaires net de chaque ligne.

**9.** Calculez :
- le chiffre d'affaires net total ;
- le nombre de lignes de commande ;
- le nombre de commandes distinctes ;
- le volume total d'articles ;
- le nombre de clients distincts.

**10.** Calculez le chiffre d'affaires par mois. Quel mois est le plus performant ? Le mois le moins performant ?

**11.** Calculez le chiffre d'affaires par canal de vente et par segment client. Utilisez une jointure avec `customers`.

**12.** Identifiez les cinq catégories et les dix produits qui génèrent le plus de chiffre d'affaires.

**À vous :** formulez trois constats chiffrés, puis deux hypothèses à vérifier.

In [ ]:
# Exercice 3.1 — indicateur de chiffre d'affaires net
# À compléter


In [ ]:
# Exercice 3.2 — KPI globaux
# À compléter


In [ ]:
# Exercice 3.3 — analyse mensuelle
# À compléter


In [ ]:
# Exercice 3.4 — canal et segment client (jointure)
# À compléter


In [ ]:
# Exercice 3.5 — catégories et produits
# À compléter


## 4. SQL avec SQLite

### Mini-cours — SQL en quelques lignes

- `SELECT` choisit les colonnes à afficher.
- `FROM` indique la table.
- `WHERE` filtre les lignes.
- `GROUP BY` regroupe les données.
- `ORDER BY` trie le résultat.
- `JOIN` relie plusieurs tables.
- `COUNT`, `SUM`, `AVG`, `MIN`, `MAX` calculent des agrégats.

En SQL, `COUNT(DISTINCT customer_id)` compte les clients uniques. `LEFT JOIN` conserve toutes les lignes de la table de gauche, même sans correspondance à droite.

### 4.1 Créer la base de données

La base est enregistrée sous `atelier_ventes.sqlite` dans le dossier courant. Les tables sont alimentées avec les données nettoyées ou de référence.

In [ ]:
# On repart des données nettoyées si vous avez terminé l'exercice 2.
# Si orders_clean n'existe pas encore, cette cellule utilise orders sans doublons.
orders_sql = orders_clean.copy() if "orders_clean" in globals() else orders.drop_duplicates().copy()
orders_sql["order_date"] = pd.to_datetime(orders_sql["order_date"]).dt.strftime("%Y-%m-%d")

db_path = Path("atelier_ventes.sqlite")
conn = sqlite3.connect(db_path)

customers.to_sql("customers", conn, if_exists="replace", index=False)
products.to_sql("products", conn, if_exists="replace", index=False)
orders_sql.to_sql("orders", conn, if_exists="replace", index=False)

print(f"Base créée : {db_path.resolve()}")
print(pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn))

### Exercices 4 — écrire vos requêtes SQL

**13.** Affichez les 10 lignes de commande les plus récentes.

**14.** Calculez le chiffre d'affaires net par canal de vente. Triez du plus élevé au plus faible.

**15.** Avec une jointure, calculez le chiffre d'affaires par région.

**16.** Affichez les catégories dont le chiffre d'affaires dépasse 100 000 €.

**17.** Comparez le nombre de commandes et le chiffre d'affaires moyen par segment client.

Écrivez vos requêtes dans les cellules suivantes. Vous pouvez ensuite charger les résultats dans pandas pour poursuivre l'analyse.

In [ ]:
# Exercice 4.1 — 10 commandes les plus récentes
query_13 = '''
-- Écrivez votre requête SQL ici
'''
# result_13 = pd.read_sql_query(query_13, conn)
# display(result_13)


In [ ]:
# Exercice 4.2 — CA par canal
query_14 = '''
-- Écrivez votre requête SQL ici
'''
# result_14 = pd.read_sql_query(query_14, conn)
# display(result_14)


In [ ]:
# Exercice 4.3 — CA par région avec JOIN
query_15 = '''
-- Écrivez votre requête SQL ici
'''
# result_15 = pd.read_sql_query(query_15, conn)
# display(result_15)


In [ ]:
# Exercice 4.4 — catégories au-dessus du seuil
query_16 = '''
-- Écrivez votre requête SQL ici
'''
# result_16 = pd.read_sql_query(query_16, conn)
# display(result_16)


In [ ]:
# Exercice 4.5 — comparaison des segments
query_17 = '''
-- Écrivez votre requête SQL ici
'''
# result_17 = pd.read_sql_query(query_17, conn)
# display(result_17)


## 5. Datavisualisation — raconter les résultats

### Mini-rappel — choisir le bon graphique
- Courbe : évolution dans le temps.
- Barres : comparer des catégories.
- Histogramme : observer la distribution d'une variable numérique.
- Nuage de points : étudier une relation entre deux variables.
- Boxplot : comparer des distributions et repérer des valeurs atypiques.

Un graphique doit avoir un titre informatif, des axes nommés, une unité et une échelle lisible. Évitez les graphiques décoratifs qui ne répondent à aucune question métier.

### Exercices 5 — visualiser et interpréter

**18.** Tracez une courbe du chiffre d'affaires mensuel. Ajoutez un titre et les unités.

**19.** Représentez le chiffre d'affaires par catégorie avec un graphique en barres, trié.

**20.** Comparez la distribution du chiffre d'affaires par commande selon le segment client (boxplot). Pour cela, agrégez d'abord les lignes par commande et ajoutez le segment client.

**21.** Choisissez une autre visualisation pertinente et expliquez la question métier à laquelle elle répond.

Pour chaque graphique, écrivez deux phrases d'interprétation : un constat et une implication possible pour l'entreprise.

In [ ]:
# Exercice 5.1 — courbe mensuelle
# À compléter avec matplotlib ou seaborn


In [ ]:
# Exercice 5.2 — CA par catégorie
# À compléter


In [ ]:
# Exercice 5.3 — distribution du CA par commande et segment
# À compléter


In [ ]:
# Exercice 5.4 — votre visualisation libre
# À compléter


## 6. Synthèse métier — votre restitution à la direction

Vous êtes maintenant chargé(e) de présenter un diagnostic court. Ne recopiez pas simplement les résultats des cellules : sélectionnez les informations utiles à une décision.

### Livrable à rédiger

Répondez aux questions suivantes dans une cellule Markdown :

1. **Performance globale :** comment se porte l'activité sur la période ?
2. **Produits :** quelles catégories ou quels produits méritent une attention particulière ?
3. **Clients et territoires :** quels segments ou quelles régions contribuent le plus au chiffre d'affaires ?
4. **Canaux :** quel canal semble le plus performant ? Attention à distinguer CA total et performance moyenne.
5. **Qualité :** quelles limites des données doivent être signalées ?
6. **Recommandations :** proposez trois actions concrètes, chacune appuyée par un résultat.
7. **Analyses complémentaires :** quelles données supplémentaires demanderiez-vous à l'entreprise ?

### Trame de restitution
- Contexte et période étudiée
- Méthode et précautions
- 3 à 5 résultats clés chiffrés
- 2 ou 3 visualisations utiles
- 3 recommandations priorisées
- Limites et prochaines analyses

## 7. Auto-évaluation et bilan de la semaine

Cochez les compétences que vous avez réellement pratiquées.

- [ ] Importer et explorer des données avec pandas
- [ ] Contrôler les valeurs manquantes et les doublons
- [ ] Nettoyer des données en justifiant ses choix
- [ ] Créer des indicateurs avec pandas
- [ ] Réaliser des jointures (`merge`)
- [ ] Écrire des requêtes SQL avec `SELECT`, `WHERE`, `GROUP BY`, `JOIN`
- [ ] Charger les résultats SQL dans pandas
- [ ] Créer des graphiques avec Matplotlib / Seaborn
- [ ] Interpréter les résultats en langage métier
- [ ] Formuler des recommandations argumentées

**Bilan personnel**
- Ce que je maîtrise :
- Ce que je dois revoir :
- L'exercice qui m'a posé le plus de difficulté :
- Ce que je pourrais expliquer à un recruteur :
- Temps réellement passé :

### Pour aller plus loin
Ne cherchez pas à tout faire parfaitement en une séance. Reprenez les exercices où vous avez hésité, puis refaites-les sans regarder vos premières tentatives. La semaine suivante pourra introduire un autre contexte métier (RH / recrutement, logistique ou santé publique), des données plus volumineuses et des analyses plus avancées.

## 8. Fermer la connexion SQL

Vous pouvez relancer le notebook plus tard : la base SQLite sera recréée à l'exécution de la section 4.1.

In [ ]:
conn.close()
print("Connexion SQLite fermée.")